# Дискретные гармоники и преобразование Фурье

Можно ли сложить несколько синусов и косинусов так, чтобы получилась кошачья голова с острыми ушами? Если задать контур значениями в конечном числе точек, сумма гармоник может в точности воспроизвести каждое из этих значений.

## 1. Что такое дискретная гармоника?

На отрезке от 0 до 1 возьмём $N$ равноотстоящих точек $x_k=k/N$, где $k=0,\ldots,N-1$. Расстояние между соседними точками равно $h=1/N$. Сетка **периодическая**. Её концы отождествлены, поэтому после последней точки снова идёт первая.

Функция $\cos(2\pi x)$ совершает на этом отрезке одно полное колебание, $\cos(4\pi x)$ — два. Целое число $n$ в формуле $\cos(2\pi n x)$ задаёт номер гармоники. При $n=0$ косинус равен единице на всём отрезке. На рисунке линии показывают косинус и синус, а маркеры — их значения в точках сетки.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    widgets = None

INK, TEAL, RUST = "#15334A", "#007C78", "#A84C26"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#A8B8BF",
    "axes.labelcolor": INK, "text.color": INK, "axes.titlesize": 12,
    "font.size": 10, "figure.facecolor": "white", "savefig.facecolor": "white",
})


def choice(figure, label, options, initial):
    if widgets is None:
        figure(initial)
        plt.show()
        return
    control = widgets.SelectionSlider(options=options, value=initial, description=label,
                                      continuous_update=False, style={"description_width": "initial"})
    out = widgets.Output()

    def draw(_=None):
        with out:
            clear_output(wait=True)
            fig = figure(control.value)
            display(fig)
            plt.close(fig)

    control.observe(draw, names="value")
    display(widgets.VBox([control, out]))
    draw()


def harmonic_figure(n):
    N = 24
    x = np.linspace(0, 1, 601)
    nodes = np.arange(N) / N
    fig, ax = plt.subplots(figsize=(8.6, 2.65), constrained_layout=True)
    ax.axhline(0, color="#BDCBD1", lw=.8)
    for f, color, line, marker, label in (
        (np.cos, TEAL, "-", "o", "Косинус"), (np.sin, RUST, "--", "s", "Синус")):
        ax.plot(x, f(2*np.pi*n*x), line, color=color, lw=1.55, label=label)
        ax.plot(nodes, f(2*np.pi*n*nodes), marker, mfc="white", mec=color, ms=4.1)
    ax.set(xlim=(-.015, 1.015), ylim=(-1.25, 1.25), yticks=[-1, 0, 1],
           xlabel="Координата x", ylabel="Значение", title=f"Гармоника n = {n} на сетке из {N} точек")
    ax.legend(loc="upper center", bbox_to_anchor=(.5,-.23), ncol=2, frameon=False)
    return fig


choice(harmonic_figure, "Номер гармоники n", list(range(24)), 3)

Обе последовательности можно записать как один вектор с комплексными компонентами. Вещественная часть каждой компоненты — косинус, мнимая — синус. Такой вектор называют **дискретной комплексной гармоникой** с номером $n$ и обозначают $\varphi^{(n)}$. По формуле Эйлера его компоненты имеют вид

$$
\varphi^{(n)}_k=\cos\left(\frac{2\pi n k}{N}\right)
+i\sin\left(\frac{2\pi n k}{N}\right)
=e^{ik\theta_n},\qquad \theta_n=\frac{2\pi n}{N},\qquad i^2=-1.
$$

Здесь $k$ — номер точки. Угол $k\theta_n$ называется **фазой** гармоники в этой точке. При переходе к следующей точке фаза увеличивается на $\theta_n$.

## 2. Оператор сдвига

**Оператор циклического сдвига** $S$ переносит весь сеточный профиль на один шаг вправо. Последнее значение при этом переходит в начало.

$$
S(U_0,U_1,\ldots,U_{N-1})=(U_{N-1},U_0,\ldots,U_{N-2}).
$$

Поэтому $(SU)_k=U_{k-1}$, причём $U_{-1}=U_{N-1}$. Для гармоники эта операция даёт

$$
(S\varphi^{(n)})_k=e^{i(k-1)\theta_n}
=e^{-i\theta_n}\,e^{ik\theta_n}
=e^{-i\theta_n}\,\varphi^{(n)}_k.
$$

Множитель $e^{-i\theta_n}$ одинаков для всех $k$. Значит, действие $S$ на **весь вектор** сводится к умножению на одно число. Такое свойство и определяет собственный вектор. Если $v\ne0$ и $Sv=\lambda v$, то $v$ называется **собственным вектором оператора** $S$, а $\lambda$ — его **собственным значением**. Для $\varphi^{(n)}$ это число равно $e^{-i\theta_n}$.

На рисунке показана вещественная часть гармоники до и после сдвига. При сдвиге на один шаг максимум перемещается вправо на $h$, высота и форма профиля сохраняются.

In [ ]:
def shift_figure(r):
    N, n = 16, 2
    h = 1 / N
    x = np.linspace(0, 1, 641)
    nodes = np.arange(N) / N
    fig, ax = plt.subplots(figsize=(8.6, 2.8), constrained_layout=True)
    ax.axhline(0, color="#BDCBD1", lw=.8)
    for shift, color, line, marker, label in (
        (0, RUST, "--", "x", "До сдвига"), (r, TEAL, "-", "o", "После сдвига")):
        ax.plot(x, np.cos(2*np.pi*n*(x-shift*h)), line, color=color, lw=1.55, label=label)
        ax.plot(nodes, np.cos(2*np.pi*n*(nodes-shift*h)), marker, color=color, mfc="white", ms=4.3)
    if r:
        left, right = 0.5, 0.5+r*h
        for peak in (left,right):
            ax.vlines(peak, 1, 1.22, color=INK, linewidth=.85)
        ax.annotate("", xy=(right, 1.19), xytext=(left,1.19),
                    arrowprops={"arrowstyle":"->","color":INK, "shrinkA":0,"shrinkB":0})
    ax.set(xlim=(-.015,1.015), ylim=(-1.15,1.30), yticks=[-1,0,1],
           xlabel="Координата x", ylabel="Вещественная часть",
           title=f"Гармоника n = 2. Сдвиг вправо на {r}/16 длины отрезка")
    ax.legend(loc="upper center", bbox_to_anchor=(.5,-.23), ncol=2, frameon=False)
    return fig


choice(shift_figure, "Сдвиг в шагах сетки", list(range(5)), 1)

## 3. Как разложить профиль на гармоники?

Пусть известны $N$ значений $U_0,\ldots,U_{N-1}$. Любой такой профиль можно представить суммой $N$ дискретных гармоник. Числа $a_n$, на которые умножаются гармоники в этой сумме, называют **коэффициентами Фурье**.

$$
U_k=\sum_{n=0}^{N-1}a_n\,e^{2\pi i n k/N}.
$$

Как найти, например, $a_2$? Умножим каждое значение $U_k$ на $e^{-2\pi i\,2k/N}$ и усредним по $k$. У второй гармоники экспоненты сократятся, и в каждом слагаемом останется $a_2$. Вклады остальных гармоник при усреднении взаимно сократятся. Например, у последовательности $1,i,-1,-i$ среднее равно нулю. Точно так же выделяется коэффициент с любым номером $n$.

$$
a_n=\frac1N\sum_{k=0}^{N-1}U_k\,e^{-2\pi i n k/N}.
$$

Вычисление этих коэффициентов называют **дискретным преобразованием Фурье**. При $n=0$ экспонента равна единице, поэтому $a_0$ — просто среднее значение профиля. На рисунках ниже столбики показывают $|a_n|$ — величину каждого коэффициента. По горизонтали стоят номера гармоник. Удобно также пользоваться отрицательными номерами. Например, $N-1$ на рисунке подписан как $-1$. Добавление $N$ к номеру даёт в узле лишь дополнительный множитель $e^{2\pi i k}=1$.

### Сумма двух косинусов

Первый профиль задан формулой $U(x)=\cos(2\pi x)+\tfrac12\cos(6\pi x)$. Первый косинус совершает на отрезке одно колебание, второй — три. Почему на рисунке четыре столбика, хотя косинусов два? Каждый косинус складывается из двух комплексных гармоник.

$$
\cos\alpha=\tfrac12 e^{i\alpha}+\tfrac12 e^{-i\alpha}.
$$

От первого косинуса получаются два коэффициента по $1/2$ при $n=\pm1$, от второго — по $1/4$ при $n=\pm3$. Если в сумму входят только гармоники с $|n|\le1$, виден первый косинус. При $|n|\le3$ восстановлен весь профиль.

In [ ]:
def profile_figure(name):
    """Три фиксированных примера на периоде длины 1."""
    N = 64
    x = np.arange(N) / N
    if name == "cosines":
        u = np.cos(2*np.pi*x) + 0.5*np.cos(6*np.pi*x)
        cutoffs = (1, 3)
        title = "Сумма двух косинусов"
    elif name == "rectangle":
        u = ((x >= 0.25) & (x < 0.75)).astype(float)
        cutoffs = (1, 9)
        title = "Прямоугольный профиль"
    elif name == "cat":
        head = np.sqrt(np.maximum(0, 0.34**2 - (x - 0.5)**2))
        ears = 0.46 * np.maximum(0, 1 - np.abs(np.abs(x - 0.5) - 0.22) / 0.09)
        u = np.maximum(head, ears)
        cutoffs = (2, 12)
        title = "Кошка"
    else:
        raise ValueError(name)

    a = np.fft.fft(u) / N
    n = np.rint(N * np.fft.fftfreq(N)).astype(int)

    def partial_sum(M):
        return np.fft.ifft(N * a * (np.abs(n) <= M)).real

    dark, green, orange = "#15334A", "#007C78", "#A84C26"
    fig, (profile, spectrum, sums) = plt.subplots(1, 3, figsize=(10.2, 3.1),
        gridspec_kw={"width_ratios": [1, 1, 1.35]}, constrained_layout=True)

    step = name == "rectangle"
    profile.fill_between(x, 0, u, color=dark, alpha=0.10,
                         step="post" if step else None)
    profile.plot(x, u, "-o", color=dark, lw=1.4, ms=2.1,
                 drawstyle="steps-post" if step else "default")
    profile.set_title(title)
    profile.set_ylabel("Значение")

    spectrum.vlines(n, 0, abs(a), color=green, lw=1.4)
    spectrum.plot(n, abs(a), ".", color=green, ms=3.4)
    spectrum.set(xlabel="Номер гармоники $n$", title="Коэффициенты $|a_n|$",
                 ylim=(-0.025 * max(abs(a)), 1.12 * max(abs(a))))
    if name == "cosines":
        spectrum.set(xlim=(-5, 5), xticks=[-3, -1, 0, 1, 3])
    else:
        spectrum.set(xlim=(-33, 32), xticks=[-32, -16, 0, 16, 31])

    full = partial_sum(N // 2)
    sums.plot(x, full, "o", mfc="white", mec=dark, ms=3.4, mew=0.9,
              label="Все гармоники")
    for M, color, style in zip(cutoffs, (orange, green), ("--", "-")):
        sums.plot(x, partial_sum(M), style, color=color, lw=1.6,
                  label=f"$|n| \\leq {M}$")
    sums.set_title("Суммы гармоник")
    sums.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=3,
                frameon=False, fontsize=8, handlelength=1.6, columnspacing=0.9)

    y_min, y_max = sums.get_ylim()
    for ax in (profile, sums):
        ax.set(xlim=(-0.012, 1), xticks=[0, .25, .5, .75, 1], xlabel="$x$",
               ylim=(y_min, y_max))
    return fig


profile_figure("cosines")
plt.show()

### Прямоугольник

На средней половине отрезка профиль равен 1, снаружи — 0. Половина из 64 значений равна единице, поэтому центральный столбик имеет высоту $1/2$. Первая пара гармоник вместе со средним даёт один плавный горб. Когда в сумму входят гармоники до $|n|=9$, видны площадка и крутые края, а возле скачков появляются небольшие колебания. Сумма всех 64 гармоник точно воспроизводит значения в точках сетки.

In [ ]:
profile_figure("rectangle")
plt.show()

### Кошка

Контур головы — верхняя полуокружность с двумя треугольными ушами. На рисунке справа сумма при $|n|\le2$ передаёт общие очертания головы. При $|n|\le12$ различимы оба уха. Кружками отмечена полная сумма в точках сетки.

In [ ]:
profile_figure("cat")
plt.show()

## 4. Когда разные гармоники выглядят одинаково?

У одного косинуса на отрезке десять колебаний, у другого — два. На сетке из 12 точек все их отсчёты совпадают. На сетке из 24 точек различие видно уже в первой точке после нуля. На двух рисунках изображены одни и те же непрерывные кривые.

In [ ]:
def plot_alias(N=24):
    """Одни и те же косинусы на двух сетках."""
    grids = (12, N)
    fig, axes = plt.subplots(
        1, len(grids), figsize=(9.8, 3.15),
        sharey=True, squeeze=False
    )
    for ax, count in zip(axes[0], grids):
        # Точки для плавных кривых. Каждая 64-я точка — узел исследуемой сетки.
        x = np.linspace(0, 1, 64 * count + 1)
        nodes = np.arange(count) * 64
        fast = np.cos(2 * np.pi * 10 * x)
        slow = np.cos(2 * np.pi * 2 * x)
        ax.axhline(0, color="#CBD5DA", lw=0.8)
        ax.plot(x, fast, color="#007C78", lw=1.5, marker="o", ms=6.5,
                mfc="white", markevery=nodes, label="10 колебаний на отрезке")
        ax.plot(x, slow, color="#A84C26", linestyle="--", lw=1.6,
                marker="x", ms=5.5, mew=1.4, markevery=nodes,
                label="2 колебания на отрезке")
        same = np.allclose(fast[nodes], slow[nodes])
        state = "отсчёты совпадают" if same else "отсчёты различаются"
        ax.set(title=f"N = {count} · {state}", xlabel="Координата x",
               xlim=(-0.02, 1.02), ylim=(-1.18, 1.18), yticks=[-1, 0, 1])
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1], ["0", "¼", "½", "¾", "1"])
    axes[0, 0].set_ylabel("Значение")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
    fig.subplots_adjust(left=0.075, right=0.985, bottom=0.30, top=0.86, wspace=0.15)
    return fig


choice(plot_alias, "Узлов на правой сетке", [12, 14, 16, 18, 20, 22, 24], 24)

Почему совпали все 12 значений? В точке $x_k=k/12$ аргументы связаны равенством

$$
2\pi\,10x_k=2\pi k-2\pi\,2x_k.
$$

Косинус не меняется при добавлении целого числа периодов и при смене знака аргумента. Поэтому $\cos(2\pi\,10x_k)=\cos(2\pi\,2x_k)$ в каждой точке этой сетки.

Для комплексных гармоник общее правило ещё короче. У номеров $n$ и $n+N$ в узлах совпадают и косинус, и синус, поскольку

$$
e^{2\pi i(n+N)k/N}=e^{2\pi i n k/N}\,e^{2\pi i k}=e^{2\pi i n k/N}.
$$

Такое совпадение отсчётов у гармоник разных частот называют **алиасингом**. В нашем примере $10=-2+12$.

**Крайний случай.** На сетке из 12 точек шестая гармоника даёт ровно две точки на колебание. Косинус даёт последовательность $1,-1,1,-1,\ldots$ Синус в тех же точках равен $\sin(\pi k)=0$. Колебание между точками есть, но все его отсчёты нулевые. Частота, при которой на один период приходятся два шага сетки, называется **частотой Найквиста**.

### Что это означает для переноса и диффузии?

Пусть два начальных профиля — те же косинусы с номерами 10 и 2. В начальный момент $t=0$ их значения на 12-точечной сетке совпадают.

**Перенос.** Уравнение $u_t+c\,u_x=0$ описывает движение профиля с постоянной скоростью $c$. Здесь $u_t$ и $u_x$ — производные по времени и координате. Если вначале профиль равен $f(x)$, то в момент $t$ он равен $f(x-ct)$. После сдвига на половину шага сетки в узлы попадают прежние значения из середин ячеек. Там два косинуса различались, и теперь в каждом узле значение одного профиля равно значению другого, взятому со знаком минус.

**Диффузия.** Уравнение $u_t=\nu u_{xx}$, где $\nu>0$, описывает сглаживание профиля. Вторая производная $u_{xx}$ у косинуса $\cos(2\pi n x)$ равна $-(2\pi n)^2\cos(2\pi n x)$. Если вначале амплитуда косинуса равна единице, то в момент $t$ она равна $e^{-\nu(2\pi n)^2t}$. У десятой гармоники показатель экспоненты по модулю в $10^2/2^2=25$ раз больше, чем у второй. На правом рисунке короткое колебание уже почти исчезло, а длинное ещё видно.

In [ ]:
def motion_figure():
    N = 12
    h = 1 / N
    nodes = np.arange(N) / N
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 2.7), sharey=True, constrained_layout=True)
    for ax, model in zip(axes, ("transport", "diffusion")):
        ax.axhline(0, color="#BDCBD1", lw=.8)
        for n, color, linestyle, marker in ((10, TEAL, "-", "o"), (2, RUST, "--", "x")):
            if model == "transport":
                values = np.cos(2*np.pi*n*(nodes-h/2))
            else:
                values = np.exp(-0.5*(2*np.pi*n*h)**2) * np.cos(2*np.pi*n*nodes)
            ax.plot(nodes, values, ls=linestyle, lw=.9, marker=marker, color=color, mfc="white",
                    ms=5.8, label=f"n = {n}")
        ax.set(xlabel="Узел k", xlim=(-.025,1-h+.025), ylim=(-1.18,1.18), yticks=[-1,0,1])
        ticks = np.array([0,2,4,6,8,10])
        ax.set_xticks(ticks*h, ticks)
        ax.set_title(r"Перенос на $h/2$" if model == "transport" else r"Диффузия при $\nu t=h^2/2$")
        ax.legend(frameon=False, loc="upper right", ncol=2, fontsize=8, handlelength=1.2)
    axes[0].set_ylabel("Значение в узле")
    return fig


motion_figure()
plt.show()